In [1]:
import numpy as np
import pandas as pd
from datasets import Dataset
from datetime import datetime

In [2]:
import sys
sys.path.append('../scripts')
import vectors

# Load dataset

In [3]:
# load dataframe
df = pd.read_csv('../data/data.csv', sep='\t', dtype=str) # they are all strings!
df

,pmid,elocationid,title,journal,year,author,affiliation,abstract
0,40944767,doi: 10.1007/s10803-025-07000-w,Psychometric Properties of the Social Responsi...,Journal of autism and developmental disorders,2025,"Fátima El-Bouhali-Abdellaoui, Núria Voltas, Pa...",Research Group on Nutrition and Mental Health ...,Earlier identification of autistic traits is c...
1,40944766,doi: 10.1007/s10803-025-07039-9,Exploring the Relationships Between Theory of ...,Journal of autism and developmental disorders,2025,"Jiaxi Li, Kathy Kar-Man Shum","Department of Psychology, The University of Ho...",This study examined friendship quality and the...
2,40944641,doi: 10.1002/hbm.70351,Flexible Reconfigurations of Brain Networks Du...,Human brain mapping,2025,"Qianying Wu, Zhihao Zhang, Ming Hsu, Andrew S ...","Helen Wills Neuroscience Institute, University...",How do large-scale brain networks dynamically ...
3,40944186,pii: 2798,Food Selectivity in Children with Autism Spect...,Nutrients,2025,"Paolo Mirizzi, Marco Esposito, Orlando Ricciar...",FUSIS MCF (Clinic Neuroscience Research and Tr...,Food selectivity is a prevalent and challengin...
4,40944170,pii: 2781,Gut Microbiota and Autism Spectrum Disorders: ...,Nutrients,2025,"Zuzanna Lewandowska-Pietruszka, Magdalena Figl...","Poznan University of Medical Sciences, Departm...",Autism spectrum disorder (ASD) is a complex ne...
...,...,...,...,...,...,...,...,...
2995,40250148,doi: 10.1016/j.yebeh.2025.110420,The prevalence of comorbidities in people with...,Epilepsy & behavior : E&B,2025,"Binx Yezhe Lin, Lisa Gong, Yifan Li, Hillary S...","Division of Addiction Science, Prevention, and...",To better understand medical comorbidity in pe...
2996,40249741,pii: e3003143,Vaccines work… and do not cause autism.,PLoS biology,2025,Nonia Pariente,"Public Library of Science, San Francisco, Cali...","Vaccines have saved millions of lives, yet the..."
2997,40249667,doi: 10.1080/17501911.2025.2491294,Mom genes and dad genes: genomic imprinting in...,Epigenomics,2025,"Erin M O'Leary, Paul J Bonthuis","Neuroscience Program, University of Illinois, ...",Genomic imprinting is an epigenetic phenomenon...
2998,40249409,doi: 10.1007/s10803-025-06815-x,Postural Control in Children with Autism Spect...,Journal of autism and developmental disorders,2025,"L Fradet, A Benchekri, R Tisserand, J-R Cazale...","Department of Child Psychiatry, Centre Hospita...",Autistic children (AT) are known to exhibit di...


# Embed data

## Preprocessing

In [4]:
# make all missing values empty strings
df = vectors.preprocess_for_vectorization(df).copy()

In [5]:
# combine text fields to be embedded
df_toembed = pd.DataFrame(df['journal']+' / '+df['title']+' / '+df['abstract'], columns=['text'])
df_toembed

,text
0,Journal of autism and developmental disorders ...
1,Journal of autism and developmental disorders ...
2,Human brain mapping / Flexible Reconfiguration...
3,Nutrients / Food Selectivity in Children with ...
4,Nutrients / Gut Microbiota and Autism Spectrum...
...,...
2995,Epilepsy & behavior : E&B / The prevalence of ...
2996,PLoS biology / Vaccines work… and do not cause...
2997,Epigenomics / Mom genes and dad genes: genomic...
2998,Journal of autism and developmental disorders ...


In [6]:
# how many words are the texts?
print(df_toembed['text'].apply(lambda x : \
len(x.split())).quantile([0.5, 0.75, 0.99, 1.0])) # percentiles

0.50    120.50
0.75    221.00
0.99    427.02
1.00    554.00
Name: text, dtype: float64


## Work, in batches

In [7]:
# vectorizer handle
print(vectors.model_handle)

multi-qa-MiniLM-L6-cos-v1


In [8]:
def batch_embed(batch):
    vectorizer = vectors.model # use locally saved model
    return {'embedding' : vectorizer.encode(batch['text'])}

In [9]:
# by batch, embed text
print(datetime.now())
dataset_to_embed = Dataset.from_pandas(df_toembed)
list_of_docs_with_embeddings = dataset_to_embed.map(batch_embed, batched=True, batch_size=32)
print(datetime.now())

2025-09-14 00:32:06.195328


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

2025-09-14 00:35:30.151881


In [10]:
# numpy format embeddings
embeddings = np.array([doc['embedding'] for doc in list_of_docs_with_embeddings])
embeddings

array([[-0.03023502,  0.01300008, -0.03873019, ...,  0.09467568,
         0.01642812,  0.04771388],
       [-0.00020004, -0.03115558, -0.07077343, ...,  0.09987616,
         0.01810992,  0.04931938],
       [-0.00641062, -0.11108153,  0.02471148, ...,  0.02989222,
         0.01300017, -0.05937385],
       ...,
       [-0.04297265, -0.00307461, -0.04133916, ...,  0.09083919,
         0.10996308, -0.06429335],
       [ 0.01068021, -0.04968933, -0.01784435, ...,  0.11307745,
        -0.00443582,  0.06951979],
       [ 0.0484339 ,  0.00486179,  0.06237775, ...,  0.05072296,
        -0.06729247,  0.10284049]], shape=(3000, 384))

# CSV file

In [11]:
# write CSV file
np.savetxt('../data/embedded.csv', embeddings, delimiter=',')

In [12]:
print(datetime.now())

2025-09-14 00:35:31.206015
